In [ ]:
# define the libraries

# for neural network model
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [6]:
# Load the CSV file
csv_file_path = "Traffic_Count_Locations_with_LONG_LAT.csv"
VictoriaData = pd.read_csv(csv_file_path)

# Display the first few rows of the dataset to verify it loaded correctly
print(VictoriaData.head())

            X          Y   FID  OBJECTID  TFM_ID  \
0  144.250614 -36.779313  7001      7301    7656   
1  145.356779 -37.835309  7002      7302   29406   
2  144.988844 -37.824629  7003      7303   22676   
3  144.932442 -37.803783  7004      7304   27902   
4  145.030601 -37.660502  7005      7305   10935   

                                TFM_DESC    TFM_TYP_DE MOVEMENT_T  \
0                CALDER HWY NE OF OAK ST  INTERSECTION  All Moves   
1  MT DANDENONG RD S BD SE OF UPALONG RD  INTERSECTION  All Moves   
2              SWAN ST W BD E OF PUNT RD  INTERSECTION  All Moves   
3        DYNON RD W BD E OF RADCLIFFE ST  INTERSECTION  All Moves   
4               DALTON RD N of CHILDS RD  INTERSECTION  All Moves   

                            SITE_DESC  ROAD_NBR            DECLARED_R  \
0                 CALDER HWY & OAK ST      2530        CALDER HIGHWAY   
1    MT DANDENONG RD SE OF UPALONG RD      4991  MOUNT DANDENONG ROAD   
2  PUNT RD LEFT TURN TO SWAN ST OD:12      2080      

In [7]:
#to filter dataset to only show data from Boroondara
BoorondaraData = VictoriaData[
    (VictoriaData['X'] > 145.0) & (VictoriaData['X'] < 145.3) &
    (VictoriaData['Y'] > -37.87) & (VictoriaData['Y'] < -37.78)
].copy()
print(BoorondaraData.shape)
print(BoorondaraData[['TFM_DESC', 'X', 'Y', 'AADT_ALLVE']].head(20))

(6975, 20)
                                        TFM_DESC           X          Y  \
8                 PRINCESS ST N OF HUTCHINSON DR  145.031281 -37.794104   
13            MAROONDAH HWY E BD E of STATION ST  145.124290 -37.818355   
15          STUDLEY PARK RD 50M AFTER HODGSON ST  145.016066 -37.805651   
16       MAROONDAH HWY SWBD SW OF BONNIE VIEW RD  145.288159 -37.780982   
21                   STUD_RD NE of KNOX CITY_ENT  145.236407 -37.866713   
24            S.E.ARTERIAL E BD W OF TOORONGA RD  145.045719 -37.848852   
28       TOORAK RD THRU MVMNT TO TOORAK RD OD:31  145.039638 -37.844746   
30                  STUDLEY PARK RD AT WALMER ST  145.011158 -37.803521   
31                 S.E.ARTERIAL E OF TOORONGA_RD  145.045774 -37.848692   
39                      BRIDGE RD E of COPPIN ST  145.003789 -37.819041   
49    SPRINGVALE RD N OF MAROONDAH HWY DEPARTURE  145.176415 -37.815848   
51                       STATION ST N OF ELEY RD  145.121065 -37.844223   
66            

In [8]:
# Replace with actual SCATS dataset filename
scats = pd.read_csv("scats_boroondara.csv")
print(scats.head())
print(scats.dtypes)

FileNotFoundError: [Errno 2] No such file or directory: 'scats_boroondara.csv'

In [ ]:

scaler = MinMaxScaler()
flow_values = scaler.fit_transform(scats[['flow']].values)

SEQ_LEN = 12  # 12 x 15min = 3 hours of history

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = make_sequences(flow_values, SEQ_LEN)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
def flow_to_speed(flow):
    # Quadratic: flow = -1.4648375 * speed^2 + 93.75 * speed
    # Solve for speed using quadratic formula
    A = -1.4648375
    B = 93.75
    discriminant = B**2 + 4 * A * flow  # note: 4*A*(-flow) since Ax^2+Bx-flow=0
    if discriminant < 0:
        return 32  # congested, return capacity speed
    speed_green = (-B + np.sqrt(B**2 + 4*A*flow)) / (2*A)
    speed_red   = (-B - np.sqrt(B**2 + 4*A*flow)) / (2*A)
    speed = max(speed_green, speed_red)
    return min(speed, 60)  # cap at speed limit

def travel_time(distance_km, flow_veh_per_hour, intersection_delay_s=30):
    speed = flow_to_speed(flow_veh_per_hour)
    time_hours = distance_km / speed
    return time_hours * 3600 + intersection_delay_s  # return in seconds